In [9]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import re
from itertools import combinations
from scipy.stats import spearmanr



In [7]:
path = "saved_files/"
dfs_boyas = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_boyas_") and archivo.endswith(".csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        df = pd.read_csv(ruta_completa)
        df = df.pivot(index='Date', columns='Buoy', values='Chl') # pivot para tener una columna para cada boya
        dfs_boyas[nombre_sin_extension] = df

In [14]:

# Asume que tienes un DataFrame tipo pivotado: fechas como índice, columnas como boyas
# Ejemplo: df_pivot = df.pivot(index='Date', columns='Buoy', values='Chl')

df = dfs_boyas["df_boyas_imida_depth_lt_1"]
resultados = []

# Generar todas las combinaciones únicas de boyas
for b1, b2 in combinations(df.columns, 2):
    # Eliminar NaNs antes de comparar
    serie1 = df[b1]
    serie2 = df[b2]
    serie_comun = pd.concat([serie1, serie2], axis=1).dropna()

    if not serie_comun.empty:
        rho, pval = spearmanr(serie_comun[b1], serie_comun[b2])
        resultados.append({'Buoy1': b1, 'Buoy2': b2, 'Spearman_rho': rho.round(3), 'p_value': pval})
    else:
        resultados.append({'Buoy1': b1, 'Buoy2': b2, 'Spearman_rho': None, 'p_value': None})

df_spearman = pd.DataFrame(resultados)
df_spearman = df_spearman.sort_values(by='Spearman_rho', ascending=False)


In [24]:
# Especifica la boya de interés
boya_objetivo = 'CTD8'

# Filtrar todas las filas donde la boya aparezca como Buoy1 o Buoy2
filtro = (df_spearman['Buoy1'] == boya_objetivo) | (df_spearman['Buoy2'] == boya_objetivo)
df_filtrado = df_spearman[filtro].copy()

# Crear una columna con la "otra boya"
df_filtrado['Otra_boya'] = df_filtrado.apply(
    lambda row: row['Buoy2'] if row['Buoy1'] == boya_objetivo else row['Buoy1'],
    axis=1
)

# Ordenar por Spearman rho descendente y mostrar las 4 más altas
df_top4 = df_filtrado.sort_values(by='Spearman_rho', ascending=False).head(4)

df_top4

,Buoy1,Buoy2,Spearman_rho,p_value,Otra_boya
29,CTD12,CTD8,0.883,5.756127e-116,CTD12
43,CTD6,CTD8,0.876,9.585362e-114,CTD6
16,CTD10,CTD8,0.874,2.449568e-111,CTD10
23,CTD11,CTD8,0.854,1.665818e-100,CTD11
